## Load and Filter Lamiales Data

⚠️ **Note**: The source file `Taxon.tsv` (2.25 GB) is **NOT** included in this repository due to its size. 

**Local file location**: `/Users/alexreese/Downloads/TEMP/backbone/Taxon.tsv` *(or wherever it actually is)*

This cell:
- Loads the full GBIF Backbone Taxonomy dataset in chunks
- Filters for:
  - Kingdom: Plantae (plants only)
  - Order: Lamiales (our taxonomic focus)
  - Status: accepted (valid names only)
- Processes ~100,000 rows at a time to manage memory
- Results in a dataframe `df` with all Lamiales taxa

**Data source**: [GBIF Backbone Taxonomy](https://www.gbif.org/dataset/d7dddbf4-2cf0-4f39-9b2a-bb099caae36c)

In [3]:
import pandas as pd

# === CONFIGURATION ===
# ⚠️ This file is NOT in the repository (too large - 2.25 GB)
TAXON_FILE_PATH = '/Users/alexreese/Downloads/TEMP/backbone/Taxon.tsv'

# Process in chunks
chunk_size = 100000
lamiales_data = []

print("Processing file in chunks...")
for i, chunk in enumerate(pd.read_csv(TAXON_FILE_PATH, sep='\t', chunksize=chunk_size, on_bad_lines='skip')):
    # Filter each chunk
    filtered_chunk = chunk[
        (chunk['kingdom'] == 'Plantae') & 
        (chunk['order'] == 'Lamiales') &
        (chunk['taxonomicStatus'] == 'accepted')
    ]
    
    if len(filtered_chunk) > 0:
        lamiales_data.append(filtered_chunk)
    
    if i % 10 == 0:
        print(f"Processed {i * chunk_size:,} rows...")

# Combine all filtered chunks
df = pd.concat(lamiales_data, ignore_index=True)
print(f"\nTotal Lamiales rows: {len(df):,}")
print(f"Columns available: {', '.join(df.columns.tolist())}")

Processing file in chunks...


<ipython-input-3-4fe0844f7b64>:12: DtypeWarning: Columns (10,16) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(TAXON_FILE_PATH, sep='\t', chunksize=chunk_size, on_bad_lines='skip')):


Processed 0 rows...
Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...


<ipython-input-3-4fe0844f7b64>:12: DtypeWarning: Columns (6,9,10,13,16,22) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(TAXON_FILE_PATH, sep='\t', chunksize=chunk_size, on_bad_lines='skip')):
<ipython-input-3-4fe0844f7b64>:12: DtypeWarning: Columns (6,7,8,9,10,13,16,21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(TAXON_FILE_PATH, sep='\t', chunksize=chunk_size, on_bad_lines='skip')):


Processed 4,000,000 rows...


<ipython-input-3-4fe0844f7b64>:12: DtypeWarning: Columns (6,7,8,9,10,13,16,22) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(TAXON_FILE_PATH, sep='\t', chunksize=chunk_size, on_bad_lines='skip')):
<ipython-input-3-4fe0844f7b64>:12: DtypeWarning: Columns (10,16) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(TAXON_FILE_PATH, sep='\t', chunksize=chunk_size, on_bad_lines='skip')):
<ipython-input-3-4fe0844f7b64>:12: DtypeWarning: Columns (9,10,16) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(TAXON_FILE_PATH, sep='\t', chunksize=chunk_size, on_bad_lines='skip')):


Processed 5,000,000 rows...
Processed 6,000,000 rows...
Processed 7,000,000 rows...

Total Lamiales rows: 42,395
Columns available: taxonID, datasetID, parentNameUsageID, acceptedNameUsageID, originalNameUsageID, scientificName, scientificNameAuthorship, canonicalName, genericName, specificEpithet, infraspecificEpithet, taxonRank, nameAccordingTo, namePublishedIn, taxonomicStatus, nomenclaturalStatus, taxonRemarks, kingdom, phylum, class, order, family, genus


In [4]:
# See the breakdown
print("Family counts:")
family_counts = df['family'].value_counts()
print(family_counts.head(20))

print("\n" + "="*50)
print(f"Total unique families: {df['family'].nunique()}")
print(f"Total unique genera: {df['genus'].nunique()}")

# Check what taxonRank values we have
print("\nTaxon rank distribution:")
print(df['taxonRank'].value_counts())

Family counts:
family
Lamiaceae           12643
Acanthaceae          6990
Gesneriaceae         5682
Orobanchaceae        3849
Plantaginaceae       3720
Scrophulariaceae     3077
Verbenaceae          1435
Bignoniaceae         1434
Oleaceae             1336
Lentibulariaceae      622
Calceolariaceae       420
Phrymaceae            376
Linderniaceae         369
Pedaliaceae           123
Mazaceae               79
Stilbaceae             64
Schlegeliaceae         44
Martyniaceae           43
Rehmanniaceae          18
Paulowniaceae          16
Name: count, dtype: int64

Total unique families: 30
Total unique genera: 1219

Taxon rank distribution:
taxonRank
species       31583
unranked       4210
subspecies     2681
variety        2545
genus          1219
form            126
family           30
order             1
Name: count, dtype: int64


In [5]:
# Keep top 10 families (gives good diversity)
top_families = family_counts.head(10).index.tolist()

print("Keeping families:", top_families)

# Filter to these families and species-level only (cleaner visualization)
df_filtered = df[
    (df['family'].isin(top_families)) & 
    (df['taxonRank'] == 'species')
].copy()

print(f"\nSpecies-level records: {len(df_filtered):,}")

# For each family, keep top 8 genera by species count
def keep_top_genera_per_family(group, n=8):
    genus_counts = group['genus'].value_counts()
    top_genera = genus_counts.head(n).index
    return group[group['genus'].isin(top_genera)]

df_filtered = df_filtered.groupby('family', group_keys=False).apply(
    lambda x: keep_top_genera_per_family(x, n=8)
)

print(f"After keeping top 8 genera per family: {len(df_filtered):,}")

# For each genus, keep top 5 species (to avoid overwhelming the viz)
def keep_top_species_per_genus(group, n=5):
    return group.head(n)

df_filtered = df_filtered.groupby(['family', 'genus'], group_keys=False).apply(
    lambda x: keep_top_species_per_genus(x, n=5)
)

print(f"After keeping top 5 species per genus: {len(df_filtered):,}")

# Show the breakdown
print("\n" + "="*50)
print("Final dataset breakdown:")
for family in top_families:
    fam_data = df_filtered[df_filtered['family'] == family]
    print(f"{family}: {fam_data['genus'].nunique()} genera, {len(fam_data)} species")

# Save it
df_filtered.to_csv('lamiales_viz_data.tsv', sep='\t', index=False)
print("\n✓ Saved to lamiales_viz_data.tsv")

Keeping families: ['Lamiaceae', 'Acanthaceae', 'Gesneriaceae', 'Orobanchaceae', 'Plantaginaceae', 'Scrophulariaceae', 'Verbenaceae', 'Bignoniaceae', 'Oleaceae', 'Lentibulariaceae']

Species-level records: 30,394
After keeping top 8 genera per family: 16,856
After keeping top 5 species per genus: 376

Final dataset breakdown:
Lamiaceae: 8 genera, 40 species
Acanthaceae: 8 genera, 40 species
Gesneriaceae: 8 genera, 40 species
Orobanchaceae: 8 genera, 40 species
Plantaginaceae: 8 genera, 40 species
Scrophulariaceae: 8 genera, 40 species
Verbenaceae: 8 genera, 40 species
Bignoniaceae: 8 genera, 40 species
Oleaceae: 8 genera, 40 species
Lentibulariaceae: 4 genera, 16 species

✓ Saved to lamiales_viz_data.tsv


<ipython-input-5-7a3bc9533cbe>:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_filtered = df_filtered.groupby('family', group_keys=False).apply(
<ipython-input-5-7a3bc9533cbe>:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_filtered = df_filtered.groupby(['family', 'genus'], group_keys=False).apply(


In [6]:
import json

# Build hierarchical structure
# Order → Family → Genus → Species

hierarchy = {
    "name": "Lamiales",
    "level": "order",
    "children": []
}

# Group by family
for family_name in df_filtered['family'].unique():
    family_data = df_filtered[df_filtered['family'] == family_name]
    
    family_node = {
        "name": family_name,
        "level": "family",
        "children": []
    }
    
    # Group by genus within this family
    for genus_name in family_data['genus'].unique():
        genus_data = family_data[family_data['genus'] == genus_name]
        
        genus_node = {
            "name": genus_name,
            "level": "genus",
            "children": []
        }
        
        # Add species
        for _, species_row in genus_data.iterrows():
            species_node = {
                "name": species_row['scientificName'],
                "level": "species",
                "canonicalName": species_row.get('canonicalName', ''),
                "taxonID": species_row['taxonID']
            }
            genus_node['children'].append(species_node)
        
        family_node['children'].append(genus_node)
    
    hierarchy['children'].append(family_node)

# Save as JSON
with open('lamiales_hierarchy.json', 'w') as f:
    json.dump(hierarchy, f, indent=2)

print("✓ Created lamiales_hierarchy.json")
print(f"  - {len(hierarchy['children'])} families")
print(f"  - Total nodes: ~{376 + 80 + 10 + 1} (species + genera + families + order)")

✓ Created lamiales_hierarchy.json
  - 10 families
  - Total nodes: ~467 (species + genera + families + order)


In [12]:
import pandas as pd
import json
import numpy as np
import re

# === CONFIGURATION ===
TOTAL_GENERA_BUDGET = 80
TOTAL_SPECIES_BUDGET = 400
MIN_GENERA_PER_FAMILY = 3
MIN_SPECIES_PER_GENUS = 2

# === Function to extract year ===
def extract_year(name_published_in):
    """Extract year in format (YYYY) from namePublishedIn field"""
    if pd.isna(name_published_in):
        return None
    match = re.search(r'\((\d{4})\)', str(name_published_in))
    if match:
        return int(match.group(1))
    return None

# Add year column
print("Extracting discovery years...")
df['discovery_year'] = df['namePublishedIn'].apply(extract_year)

species_with_years = df[df['taxonRank'] == 'species']['discovery_year'].notna().sum()
total_species = len(df[df['taxonRank'] == 'species'])
print(f"Found years for {species_with_years:,} out of {total_species:,} species ({species_with_years/total_species*100:.1f}%)")

# === STEP 1: Calculate actual genus counts per family ===
print("\nStep 1: Calculating actual proportions...")
family_genus_counts = df[
    (df['family'].isin(top_families)) & 
    (df['taxonRank'].isin(['species', 'genus']))
].groupby('family')['genus'].nunique()

print("\nActual genus counts per family:")
print(family_genus_counts.sort_values(ascending=False))

# === STEP 2: Allocate genera budget proportionally ===
print("\nStep 2: Allocating genera budget proportionally...")
total_actual_genera = family_genus_counts.sum()
genera_allocation = {}

for family in top_families:
    actual_count = family_genus_counts.get(family, 0)
    proportion = actual_count / total_actual_genera
    allocated = max(MIN_GENERA_PER_FAMILY, int(proportion * TOTAL_GENERA_BUDGET))
    genera_allocation[family] = allocated

# Normalize to exactly hit budget
current_total = sum(genera_allocation.values())
if current_total != TOTAL_GENERA_BUDGET:
    largest_family = family_genus_counts.idxmax()
    genera_allocation[largest_family] += (TOTAL_GENERA_BUDGET - current_total)

print("\nAllocated genera per family:")
for family, count in sorted(genera_allocation.items(), key=lambda x: x[1], reverse=True):
    actual = family_genus_counts.get(family, 0)
    print(f"  {family}: {count} genera (actual: {actual})")

# === STEP 3: Sample genera and prioritize early discoveries ===
print("\nStep 3: Sampling genera and prioritizing early discoveries...")
sampled_data = []

for family in top_families:
    family_df = df[
        (df['family'] == family) & 
        (df['taxonRank'] == 'species')
    ].copy()
    
    if len(family_df) == 0:
        continue
    
    genus_counts = family_df['genus'].value_counts()
    n_genera = genera_allocation[family]
    selected_genera = genus_counts.head(n_genera).index.tolist()
    
    family_df = family_df[family_df['genus'].isin(selected_genera)]
    
    # Allocate species proportionally within this family
    family_species_budget = int((len(family_df) / len(df[df['family'].isin(top_families)])) * TOTAL_SPECIES_BUDGET)
    family_species_budget = max(n_genera * MIN_SPECIES_PER_GENUS, family_species_budget)
    
    genus_species_counts = family_df['genus'].value_counts()
    total_in_family = genus_species_counts.sum()
    
    for genus in selected_genera:
        actual_species_count = genus_species_counts.get(genus, 0)
        proportion = actual_species_count / total_in_family
        n_species = max(MIN_SPECIES_PER_GENUS, int(proportion * family_species_budget))
        
        genus_df = family_df[family_df['genus'] == genus].copy()
        
        # Sort by discovery year (earliest first)
        genus_df = genus_df.sort_values('discovery_year', na_position='last')
        genus_df = genus_df.head(n_species)
        sampled_data.append(genus_df)

# Combine
df_final = pd.concat(sampled_data, ignore_index=True)

print(f"\nFinal dataset: {len(df_final)} species")
print(f"Species with discovery years: {df_final['discovery_year'].notna().sum()}")
if df_final['discovery_year'].notna().sum() > 0:
    print(f"Year range: {df_final['discovery_year'].min():.0f} - {df_final['discovery_year'].max():.0f}")

print("\nBreakdown by family:")
for family in top_families:
    fam_data = df_final[df_final['family'] == family]
    if len(fam_data) > 0:
        years = fam_data['discovery_year'].dropna()
        year_info = f", years: {years.min():.0f}-{years.max():.0f}" if len(years) > 0 else ""
        print(f"  {family}: {fam_data['genus'].nunique()} genera, {len(fam_data)} species{year_info}")

# === Calculate ACTUAL totals for context ===
print("\nCalculating actual totals for transparency...")

# Total families in Lamiales
actual_families_total = len(top_families)

# Actual genera per family
actual_genera_per_family = {}
for family in top_families:
    actual_genera_per_family[family] = df[
        (df['family'] == family) & 
        (df['taxonRank'].isin(['species', 'genus']))
    ]['genus'].nunique()

# Actual species per genus (for displayed genera)
actual_species_per_genus = {}
for family in top_families:
    family_genera = df_final[df_final['family'] == family]['genus'].unique()
    for genus in family_genera:
        actual_species_per_genus[genus] = len(df[
            (df['genus'] == genus) & 
            (df['taxonRank'] == 'species')
        ])

print(f"Total families in Lamiales: {actual_families_total}")
print(f"Example - Lamiaceae actual genera: {actual_genera_per_family.get('Lamiaceae', 'N/A')}")

# === STEP 4 UPDATED: Build hierarchical JSON with actual counts ===
print("\nStep 4: Building hierarchical JSON with actual counts...")

hierarchy = {
    "name": "Lamiales",
    "level": "order",
    "actualChildCount": actual_families_total,  # ADD THIS
    "children": []
}

for family in top_families:
    family_data = df_final[df_final['family'] == family]
    if len(family_data) == 0:
        continue
    
    family_node = {
        "name": family,
        "level": "family",
        "actualChildCount": actual_genera_per_family.get(family, 0),  # ADD THIS
        "children": []
    }
    
    for genus in family_data['genus'].unique():
        genus_data = family_data[family_data['genus'] == genus]
        
        genus_node = {
            "name": genus,
            "level": "genus",
            "actualChildCount": actual_species_per_genus.get(genus, 0),  # ADD THIS
            "children": []
        }
        
        for _, row in genus_data.iterrows():
            species_node = {
                "name": row.get('canonicalName', row['scientificName']),
                "level": "species",
                "canonicalName": row.get('canonicalName', row['scientificName']),
                "taxonID": int(row['taxonID']),
                "discoveryYear": int(row['discovery_year']) if pd.notna(row['discovery_year']) else None,
                "authorship": row.get('scientificNameAuthorship', '')
            }
            genus_node['children'].append(species_node)
        
        family_node['children'].append(genus_node)
    
    hierarchy['children'].append(family_node)

# Save
with open('lamiales_hierarchy_proportional.json', 'w') as f:
    json.dump(hierarchy, f, indent=2)

print("\n✓ Saved to lamiales_hierarchy_proportional.json")
print(f"\nTotal nodes: {len(df_final) + df_final['genus'].nunique() + len(top_families) + 1}")

# Show examples
print("\n" + "="*60)
print("Sample of earliest discoveries:")
early = df_final.sort_values('discovery_year').head(10)
for _, row in early.iterrows():
    year = f"({row['discovery_year']:.0f})" if pd.notna(row['discovery_year']) else "(year unknown)"
    print(f"  {row['canonicalName']} {year} - {row['genus']} ({row['family']})")

Extracting discovery years...
Found years for 27,878 out of 31,583 species (88.3%)

Step 1: Calculating actual proportions...

Actual genus counts per family:
family
Lamiaceae           259
Acanthaceae         217
Gesneriaceae        172
Orobanchaceae       107
Plantaginaceae      104
Bignoniaceae        103
Scrophulariaceae     81
Verbenaceae          43
Oleaceae             42
Lentibulariaceae      4
Name: genus, dtype: int64

Step 2: Allocating genera budget proportionally...

Allocated genera per family:
  Lamiaceae: 18 genera (actual: 259)
  Acanthaceae: 15 genera (actual: 217)
  Gesneriaceae: 12 genera (actual: 172)
  Orobanchaceae: 7 genera (actual: 107)
  Plantaginaceae: 7 genera (actual: 104)
  Bignoniaceae: 7 genera (actual: 103)
  Scrophulariaceae: 5 genera (actual: 81)
  Verbenaceae: 3 genera (actual: 43)
  Oleaceae: 3 genera (actual: 42)
  Lentibulariaceae: 3 genera (actual: 4)

Step 3: Sampling genera and prioritizing early discoveries...

Final dataset: 208 species
Speci

## Generate Latest Discovered Dataset

This creates an alternative dataset prioritizing **most recently discovered species** (newest first).

**Why this matters:**
- Highlights modern botanical work and contemporary scientists
- Shows recent field discoveries (often from underexplored regions)
- Provides counterbalance to colonial-era Linnaean tradition
- Demonstrates ongoing taxonomic research

**Output**: `lamiales_hierarchy_latest.json`

**Comparison with earliest:**
- Earliest: Prioritizes 1700s-1800s discoveries (Linnaeus, early European botanists)
- Latest: Prioritizes 1900s-2000s discoveries (modern field work, diverse regions)

In [13]:
# === Generate LATEST Discovered Version ===
print("\n" + "="*60)
print("Generating LATEST discovered dataset...")

sampled_data_latest = []

for family in top_families:
    family_df = df[
        (df['family'] == family) & 
        (df['taxonRank'] == 'species')
    ].copy()
    
    if len(family_df) == 0:
        continue
    
    genus_counts = family_df['genus'].value_counts()
    n_genera = genera_allocation[family]
    selected_genera = genus_counts.head(n_genera).index.tolist()
    
    family_df = family_df[family_df['genus'].isin(selected_genera)]
    
    family_species_budget = int((len(family_df) / len(df[df['family'].isin(top_families)])) * TOTAL_SPECIES_BUDGET)
    family_species_budget = max(n_genera * MIN_SPECIES_PER_GENUS, family_species_budget)
    
    genus_species_counts = family_df['genus'].value_counts()
    total_in_family = genus_species_counts.sum()
    
    for genus in selected_genera:
        actual_species_count = genus_species_counts.get(genus, 0)
        proportion = actual_species_count / total_in_family
        n_species = max(MIN_SPECIES_PER_GENUS, int(proportion * family_species_budget))
        
        genus_df = family_df[family_df['genus'] == genus].copy()
        
        # === DIFFERENT: Sort by discovery year (LATEST first) ===
        genus_df = genus_df.sort_values('discovery_year', ascending=False, na_position='last')
        genus_df = genus_df.head(n_species)
        sampled_data_latest.append(genus_df)

# Combine
df_latest = pd.concat(sampled_data_latest, ignore_index=True)

print(f"\nLatest dataset: {len(df_latest)} species")
print(f"Species with discovery years: {df_latest['discovery_year'].notna().sum()}")
if df_latest['discovery_year'].notna().sum() > 0:
    print(f"Year range: {df_latest['discovery_year'].min():.0f} - {df_latest['discovery_year'].max():.0f}")


# === Calculate ACTUAL totals for context ===
print("\nCalculating actual totals for transparency...")

# Total families in Lamiales
actual_families_total = len(top_families)

# Actual genera per family
actual_genera_per_family = {}
for family in top_families:
    actual_genera_per_family[family] = df[
        (df['family'] == family) & 
        (df['taxonRank'].isin(['species', 'genus']))
    ]['genus'].nunique()

# Actual species per genus (for displayed genera)
actual_species_per_genus = {}
for family in top_families:
    family_genera = df_final[df_final['family'] == family]['genus'].unique()
    for genus in family_genera:
        actual_species_per_genus[genus] = len(df[
            (df['genus'] == genus) & 
            (df['taxonRank'] == 'species')
        ])

print(f"Total families in Lamiales: {actual_families_total}")
print(f"Example - Lamiaceae actual genera: {actual_genera_per_family.get('Lamiaceae', 'N/A')}")


# === STEP 4 UPDATED: Build hierarchical JSON with actual counts ===
print("\nStep 4: Building hierarchical JSON with actual counts...")

hierarchy = {
    "name": "Lamiales",
    "level": "order",
    "actualChildCount": actual_families_total,  # ADD THIS
    "children": []
}

for family in top_families:
    family_data = df_final[df_final['family'] == family]
    if len(family_data) == 0:
        continue
    
    family_node = {
        "name": family,
        "level": "family",
        "actualChildCount": actual_genera_per_family.get(family, 0),  # ADD THIS
        "children": []
    }
    
    for genus in family_data['genus'].unique():
        genus_data = family_data[family_data['genus'] == genus]
        
        genus_node = {
            "name": genus,
            "level": "genus",
            "actualChildCount": actual_species_per_genus.get(genus, 0),  # ADD THIS
            "children": []
        }
        
        for _, row in genus_data.iterrows():
            species_node = {
                "name": row.get('canonicalName', row['scientificName']),
                "level": "species",
                "canonicalName": row.get('canonicalName', row['scientificName']),
                "taxonID": int(row['taxonID']),
                "discoveryYear": int(row['discovery_year']) if pd.notna(row['discovery_year']) else None,
                "authorship": row.get('scientificNameAuthorship', '')
            }
            genus_node['children'].append(species_node)
        
        family_node['children'].append(genus_node)
    
    hierarchy['children'].append(family_node)

# Save latest
with open('lamiales_hierarchy_latest.json', 'w') as f:
    json.dump(hierarchy_latest, f, indent=2)

print("\n✓ Saved to lamiales_hierarchy_latest.json")

# Show comparison
print("\n" + "="*60)
print("COMPARISON: Earliest vs Latest")
print("\nEarliest discoveries (sample):")
early = df_final.sort_values('discovery_year').head(5)
for _, row in early.iterrows():
    year = f"({row['discovery_year']:.0f})" if pd.notna(row['discovery_year']) else "(unknown)"
    print(f"  {row['canonicalName']} {year}")

print("\nLatest discoveries (sample):")
late = df_latest.sort_values('discovery_year', ascending=False).head(5)
for _, row in late.iterrows():
    year = f"({row['discovery_year']:.0f})" if pd.notna(row['discovery_year']) else "(unknown)"
    print(f"  {row['canonicalName']} {year}")


Generating LATEST discovered dataset...

Latest dataset: 208 species
Species with discovery years: 208
Year range: 1954 - 2023

Calculating actual totals for transparency...
Total families in Lamiales: 10
Example - Lamiaceae actual genera: 259

Step 4: Building hierarchical JSON with actual counts...

✓ Saved to lamiales_hierarchy_latest.json

COMPARISON: Earliest vs Latest

Earliest discoveries (sample):
  Salvia viridis (1753)
  Pedicularis hirsuta (1753)
  Pedicularis comosa (1753)
  Pedicularis sceptrum-carolinum (1753)
  Pedicularis tristis (1753)

Latest discoveries (sample):
  Salvia adiyamanensis (2023)
  Aeschynanthus smaragdinus (2023)
  Strobilanthes sripadensis (2023)
  Strobilanthes spathulatibracteata (2023)
  Ruellia eriocalyx (2023)
